# Phase 1 — SQL Analytics Layer
## Healthcare AI System

**Objective:** Load raw hospital CSVs into a relational SQLite database 
and run operational + financial analytics queries.

**Tables:**
- `patients` — 5,000 rows
- `visits`   — 25,000 rows  
- `billing`  — 25,000 rows

In [1]:
# Imports
import pandas as pd
import sqlite3
import os

In [3]:
# 1) Load raw CSV Files
patients = pd.read_csv("../data/Corrected-Data/patients.csv")
visits = pd.read_csv("../data/Corrected-Data/visits.csv")
billing = pd.read_csv("../data/Corrected-Data/billing.csv")

print("patients shape: ", patients.shape)
print("visits shape: ", visits.shape)
print("billing shape: ", billing.shape)

patients shape:  (5000, 7)
visits shape:  (25000, 8)
billing shape:  (25000, 7)


In [4]:
# 2) Create Sqlite db
# Create db folder, if it doesn't exits
os.makedirs("../db", exist_ok=True)

# Connect to sqlite and then create db
conn = sqlite3.connect("../db/hospital.db")
cursor = conn.cursor()
print("Database connected:", conn)

Database connected: <sqlite3.Connection object at 0x110d2d4e0>


In [5]:
# 3) - Load dataframes into sqlite tables
# write all three dataframes into sqlite tables
patients.to_sql("patients", conn, if_exists="replace", index=False)
visits.to_sql("visits", conn, if_exists="replace", index=False)
billing.to_sql("billing", conn, if_exists="replace", index=False)

print("All three tables loaded into hospital db")

All three tables loaded into hospital db


In [6]:
# 4) Quick preview of tables
print("=== PATIENTS (first 3 rows) ===")
print(pd.read_sql("select * from patients limit 3", conn).to_string())

print("=== VISITS (first 3 rows) ===")
print(pd.read_sql("select * from visits limit 3", conn).to_string())

print("=== BILLING (first 3 rows) ===")
print(pd.read_sql("select * from billing limit 3", conn).to_string())

=== PATIENTS (first 3 rows) ===
   patient_id  age gender       city insurance_provider  chronic_flag registration_date
0           1   52      M  Bangalore            CareOne             1          5/6/2025
1           2   15      F     Mumbai            CareOne             0        12/27/2025
2           3   72      F  Bangalore         SecureLife             1         1/28/2025
=== VISITS (first 3 rows) ===
   visit_id  patient_id  visit_date  department visit_type  length_of_stay_hours risk_score  doctor_id
0         1           1  10/16/2025  Cardiology        OPD                 23.15        Low        186
1         2           1   12/9/2025         ICU        ICU                 39.98       High        149
2         3           1  10/29/2025          ER         ER                 20.97     Medium        122
=== BILLING (first 3 rows) ===
   bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
0        1         1       23550.00         21429.

## Operational + Financial Analytics

In [7]:
# 5) Department workload analysis
dept_workload = pd.read_sql("""
SELECT 
    department,
    COUNT(visit_id)                             AS total_visits,
    ROUND(AVG(length_of_stay_hours), 2)         AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours), 2)         AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)                      AS high_risk_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
""", conn)

print(dept_workload.to_string(index=False))

 department  total_visits  avg_los_hours  max_los_hours  high_risk_visits
    General          5757          14.01          36.54                24
Orthopedics          4474          23.02          52.99                54
 Cardiology          4071          32.59          70.74               489
         ER          3896          11.95          30.78               139
        ICU          3415          56.86          78.42              2975
  Neurology          3387          27.69          66.16               635


In [8]:
# 6) Doctor Risk Cases
# Which doctors handle the most high-risk patients?
doctor_risk = pd.read_sql("""
    SELECT 
        doctor_id,
        COUNT(visit_id)                              AS total_visits,
        SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END)                       AS high_risk_cases,
        ROUND(
            100.0 * SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END) / COUNT(visit_id), 1) AS high_risk_pct
    FROM visits
    GROUP BY doctor_id
    ORDER BY high_risk_cases DESC
    LIMIT 10
""", conn)

print("Top 10 Doctors by High Risk Cases")
print(doctor_risk.to_string(index=False))

Top 10 Doctors by High Risk Cases
 doctor_id  total_visits  high_risk_cases  high_risk_pct
       114           260               61           23.5
       130           264               60           22.7
       187           262               57           21.8
       126           244               55           22.5
       102           271               55           20.3
       189           240               54           22.5
       158           252               53           21.0
       171           260               52           20.0
       150           252               52           20.6
       194           267               51           19.1


In [9]:
# 7) Patient Visit Patterns
# How many visits does each patient make on average?
visit_patterns = pd.read_sql("""
    SELECT 
        visit_frequency_bucket,
        COUNT(patient_id) AS num_patients
    FROM (
        SELECT 
            patient_id,
            COUNT(visit_id) AS visit_count,
            CASE 
                WHEN COUNT(visit_id) = 1     THEN '1 visit'
                WHEN COUNT(visit_id) BETWEEN 2 AND 3 THEN '2-3 visits'
                WHEN COUNT(visit_id) BETWEEN 4 AND 5 THEN '4-5 visits'
                ELSE '6+ visits'
            END AS visit_frequency_bucket
        FROM visits
        GROUP BY patient_id
    )
    GROUP BY visit_frequency_bucket
    ORDER BY num_patients DESC
""", conn)

print("Patient Visit Frequency Distribution")
print(visit_patterns.to_string(index=False))

Patient Visit Frequency Distribution
visit_frequency_bucket  num_patients
             6+ visits          1816
            4-5 visits          1776
            2-3 visits          1267
               1 visit           141


In [10]:
# 8) Risk Score Distribution by Department
risk_by_dept = pd.read_sql("""
    SELECT 
        department,
        SUM(CASE WHEN risk_score = 'Low'    THEN 1 ELSE 0 END) AS low,
        SUM(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
        SUM(CASE WHEN risk_score = 'High'   THEN 1 ELSE 0 END) AS high,
        COUNT(*) AS total
    FROM visits
    GROUP BY department
    ORDER BY high DESC
""", conn)

print("Risk Score Distribution by Department")
print(risk_by_dept.to_string(index=False))

Risk Score Distribution by Department
 department  low  medium  high  total
        ICU   10     430  2975   3415
  Neurology  875    1877   635   3387
 Cardiology 1063    2519   489   4071
         ER 1771    1986   139   3896
Orthopedics 3426     994    54   4474
    General 4590    1143    24   5757


## Financial Analytics

In [11]:
# 9) Insurance Billing Breakdown
# Revenue and claim outcomes by insurance provider.
insurance_billing = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                         AS total_claims,
        ROUND(SUM(b.billed_amount), 0)           AS total_billed,
        ROUND(AVG(b.billed_amount), 0)           AS avg_billed,
        ROUND(SUM(b.approved_amount), 0)         AS total_approved,
        SUM(CASE WHEN b.claim_status = 'Paid'     
            THEN 1 ELSE 0 END)                   AS paid,
        SUM(CASE WHEN b.claim_status = 'Pending'  
            THEN 1 ELSE 0 END)                   AS pending,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                   AS rejected
    FROM billing b
    JOIN visits  v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY total_billed DESC
""", conn)

print("Insurance Billing Breakdown")
print(insurance_billing.to_string(index=False))

Insurance Billing Breakdown
insurance_provider  total_claims  total_billed  avg_billed  total_approved  paid  pending  rejected
        HealthPlus          6499   251723095.0     38733.0     124033289.0  3727     1557      1215
           CareOne          6326   246192121.0     38918.0     107084590.0  3160     1541      1625
        SecureLife          6197   242381528.0     39113.0     128230940.0  3723     1498       976
         MediCareX          5978   229000682.0     38307.0     100849491.0  3028     1500      1450


In [12]:
# 10) Claim Rejection Analysis
# Which insurance providers reject the most claims?
rejection_analysis = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        COUNT(b.bill_id)                              AS total_claims,
        SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END)                        AS rejected_claims,
        ROUND(
            100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' 
            THEN 1 ELSE 0 END) / COUNT(b.bill_id), 1) AS rejection_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY rejection_rate_pct DESC
""", conn)

print("Claim Rejection Rate by Insurance Provider")
print(rejection_analysis.to_string(index=False))

Claim Rejection Rate by Insurance Provider
insurance_provider  total_claims  rejected_claims  rejection_rate_pct
           CareOne          6326             1625                25.7
         MediCareX          5978             1450                24.3
        HealthPlus          6499             1215                18.7
        SecureLife          6197              976                15.7


In [13]:
# 11) Revenue Realization
# How much of what we bill actually gets approved?
revenue_realization = pd.read_sql("""
    SELECT 
        p.insurance_provider,
        ROUND(SUM(b.billed_amount), 0)                AS total_billed,
        ROUND(SUM(b.approved_amount), 0)              AS total_approved,
        ROUND(
            100.0 * SUM(b.approved_amount) / 
            SUM(b.billed_amount), 1)                  AS realization_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE b.approved_amount IS NOT NULL
    GROUP BY p.insurance_provider
    ORDER BY realization_rate_pct DESC
""", conn)

print("Revenue Realization Rate by Insurance Provider")
print(revenue_realization.to_string(index=False))

Revenue Realization Rate by Insurance Provider
insurance_provider  total_billed  total_approved  realization_rate_pct
        SecureLife   181215374.0     128230940.0                  70.8
        HealthPlus   180661516.0     124033289.0                  68.7
         MediCareX   157807601.0     100849491.0                  63.9
           CareOne   169349090.0     107084590.0                  63.2


## Data Quality 

In [14]:
# 12) Missing Records Check
print("=== MISSING VALUES CHECK ===\n")

for table in ['patients', 'visits', 'billing']:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"{table}:")
        print(nulls.to_string())
        print()
    else:
        print(f"{table}: ✓ No missing values\n")

=== MISSING VALUES CHECK ===

patients: ✓ No missing values

visits: ✓ No missing values

billing:
approved_amount    6156
payment_days       9532



In [15]:
# 13) Duplicate Patients Check
duplicate_patients = pd.read_sql("""
    SELECT 
        patient_id,
        COUNT(*) AS occurrences
    FROM patients
    GROUP BY patient_id
    HAVING COUNT(*) > 1
""", conn)

if len(duplicate_patients) == 0:
    print("✓ No duplicate patient_ids found")
else:
    print(f"⚠ Found {len(duplicate_patients)} duplicate patient_ids")
    print(duplicate_patients)

✓ No duplicate patient_ids found


In [16]:
# 14) Orphan Records Check
# Visits without matching patients
orphan_visits = pd.read_sql("""
    SELECT COUNT(*) AS orphan_visits
    FROM visits v
    LEFT JOIN patients p ON v.patient_id = p.patient_id
    WHERE p.patient_id IS NULL
""", conn)

# Billing without matching visits
orphan_billing = pd.read_sql("""
    SELECT COUNT(*) AS orphan_billing
    FROM billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    WHERE v.visit_id IS NULL
""", conn)

print("Orphan visits  (no matching patient):", 
      orphan_visits['orphan_visits'][0])
print("Orphan billing (no matching visit)  :", 
      orphan_billing['orphan_billing'][0])

Orphan visits  (no matching patient): 0
Orphan billing (no matching visit)  : 0


In [17]:
# 15) Invalid LOS Check
invalid_los = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN length_of_stay_hours <= 0  THEN 1 END) AS zero_or_negative,
        COUNT(CASE WHEN length_of_stay_hours > 720 THEN 1 END) AS over_30_days,
        MIN(length_of_stay_hours)                              AS min_los,
        MAX(length_of_stay_hours)                             AS max_los,
        ROUND(AVG(length_of_stay_hours), 2)                   AS avg_los
    FROM visits
""", conn)

print("LOS Validation:")
print(invalid_los.to_string(index=False))

LOS Validation:
 zero_or_negative  over_30_days  min_los  max_los  avg_los
                0             0      0.5    78.42    26.03


In [18]:
# 16) Invalid Payment Days Check
invalid_payment = pd.read_sql("""
    SELECT 
        COUNT(CASE WHEN payment_days < 0   THEN 1 END) AS negative_days,
        COUNT(CASE WHEN payment_days > 365 THEN 1 END) AS over_1_year,
        COUNT(CASE WHEN payment_days IS NULL THEN 1 END) AS nulls,
        MIN(payment_days)                               AS min_days,
        MAX(payment_days)                               AS max_days
    FROM billing
""", conn)

print("Payment Days Validation:")
print(invalid_payment.to_string(index=False))

Payment Days Validation:
 negative_days  over_1_year  nulls  min_days  max_days
             0            0   9532       1.0      55.0


## Export model_table.csv

In [19]:
# Join all three tables into one flat table for ML modeling.
# This is the single source of truth for Phase 2 EDA and Phase 3 Modeling.
model_table = pd.read_sql("""
    SELECT
        -- Patient features
        p.patient_id,
        p.age,
        p.gender,
        p.city,
        p.insurance_provider,
        p.chronic_flag,
        p.registration_date,

        -- Visit features
        v.visit_id,
        v.visit_date,
        v.department,
        v.visit_type,
        v.length_of_stay_hours,
        v.risk_score,
        v.doctor_id,

        -- Billing features
        b.bill_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.payment_days,
        b.billing_date

    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    JOIN billing  b ON v.visit_id   = b.visit_id
    ORDER BY v.visit_date
""", conn)

print("model_table shape:", model_table.shape)
print("Columns:", model_table.columns.tolist())
model_table.head(3)

model_table shape: (25000, 20)
Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,2,15,F,Mumbai,CareOne,0,12/27/2025,8,1/1/2026,General,OPD,9.63,Low,105,8,9612.77,NaN,Rejected,NaN,2026-01-19
1,12,3,M,Bangalore,CareOne,0,8/13/2025,65,1/1/2026,ICU,ICU,59.60,High,112,65,88539.01,NaN,Rejected,NaN,2026-01-05
2,129,44,M,Pune,MediCareX,1,7/20/2025,651,1/1/2026,ICU,ER,59.28,High,150,651,88539.01,NaN,Pending,NaN,2026-01-20


In [20]:
# save the table
import os
os.makedirs("../outputs", exist_ok=True)
model_table.to_csv("../outputs/model_table.csv", index=False)
print(f"model_table.csv saved --> /outputs/model_table.csv")
print(f"Shape: {model_table.shape}")
print(f"Size: {os.path.getsize('../outputs/model_table.csv')/1024:.1f} KB")

model_table.csv saved --> /outputs/model_table.csv
Shape: (25000, 20)
Size: 3025.1 KB


In [21]:
# Close Connection
conn.close()
print("Database connection closed successfully")

Database connection closed successfully
